# Audio embeddings (WavLM-large) for the Grammar Scoring Engine

This notebook turns every train and test clip into an **audio embedding**: the internal representation of the pretrained speech model WavLM-large (`microsoft/wavlm-large`, 24 layers), pooled over time. It encodes how the speech sounds (pronunciation, rhythm, hesitation, delivery), which the transcript-based features capture only partly.

For every layer, two vectors are kept: the **mean** over time (the typical sound of the clip) and the **standard deviation** over time (how much it varies, e.g. hesitations and uneven pacing).

The output is one file, `audio_embeddings_wavlm_large.npz`, with one row per clip and, per layer, `[mean (1024) | std (1024)]`. The modeling notebook loads it, chooses layers by cross-validation (or averages all of them if that is better) and fits models on it. 

**Settings:** Accelerator **GPU T4 x2**, Internet **on** (the model weights are downloaded from Hugging Face). Expected runtime: roughly 20–30 minutes.

## 1. Configuration and data

In [1]:
from math import gcd
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import resample_poly

MODEL_NAME = "microsoft/wavlm-large"
SR = 16000  # WavLM expects 16 kHz mono audio
INPUT_ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working")
OUT_PATH = OUT_DIR / "audio_embeddings_wavlm_large.npz"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
csv_candidates = sorted(INPUT_ROOT.rglob("train.csv"))
assert len(csv_candidates) == 1, f"Expected exactly one train.csv under {INPUT_ROOT}, found: {csv_candidates}"
DATA_DIR = csv_candidates[0].parent


def index_audio(folder):
    files = [p for p in folder.rglob("*") if p.suffix.lower() == ".wav"]
    index = {p.stem: p for p in files}
    assert len(index) == len(files), f"Two files share a name (ignoring extension) in {folder}"
    return index


rows = []
for split in ["train", "test"]:
    df = pd.read_csv(DATA_DIR / f"{split}.csv", dtype={"filename": str})
    audio = index_audio(DATA_DIR / split)
    for name in df["filename"]:
        file_id = Path(name).stem
        assert file_id in audio, f"No audio file for {split}/{name}"
        rows.append({"split": split, "file_id": file_id, "path": audio[file_id]})
manifest = pd.DataFrame(rows)
assert not manifest.duplicated(["split", "file_id"]).any()
print(manifest["split"].value_counts().to_string())

split
train    769
test     216


In [3]:
def load_audio_16k(path):
    audio, sr = sf.read(str(path), dtype="float32", always_2d=True)  # (samples, channels)
    audio = audio.mean(axis=1)                                       # stereo -> mono
    if sr != SR:
        g = gcd(sr, SR)
        audio = resample_poly(audio, SR // g, sr // g)
    return np.ascontiguousarray(audio, dtype=np.float32)

# Check the loader on one file
_p = manifest["path"].iloc[0]
_a = load_audio_16k(_p)
_info = sf.info(str(_p))
assert abs(len(_a) / SR - _info.duration) < 0.05, "Loaded duration doesn't match the file header"
assert np.isfinite(_a).all()
print(f"Loader OK: {len(_a) / SR:.2f}s (header {_info.duration:.2f}s), native {_info.samplerate} Hz")

Loader OK: 60.51s (header 60.51s), native 16000 Hz


## 2. Load WavLM

In [4]:
import torch
from transformers import AutoFeatureExtractor, WavLMModel

device = "cuda" if torch.cuda.is_available() else "cpu"

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
model = WavLMModel.from_pretrained(MODEL_NAME).to(device).eval()
print(f"Loaded {MODEL_NAME} on {device}: {model.config.num_hidden_layers} layers, "
      f"{model.config.hidden_size}-dim")

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.22k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.26GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

Loaded microsoft/wavlm-large on cuda: 24 layers, 1024-dim


## 3. Extract embeddings
For each clip, WavLM produces one vector per 20 ms of audio at every layer. The mean and standard deviation over time give one fixed-size vector per layer, regardless of clip length. All layers are saved; the modeling notebook decides how to combine them.

In [5]:
@torch.no_grad()
def embed_audio(path):
    audio = load_audio_16k(path)
    inputs = feature_extractor(audio, sampling_rate=SR, return_tensors="pt")
    outputs = model(inputs["input_values"].to(device), output_hidden_states=True)
    hidden = torch.stack(outputs.hidden_states).squeeze(1).float()  # (layers + 1, frames, dim)
    mean = hidden.mean(dim=1)
    std = hidden.std(dim=1, unbiased=False)
    return torch.cat([mean, std], dim=-1).cpu().numpy()            # (layers + 1, 2 * dim): [mean | std]


# Smoke test on one file before the full run.
probe = embed_audio(manifest["path"].iloc[0])
DIM = model.config.hidden_size
assert probe.shape == (model.config.num_hidden_layers + 1, 2 * DIM) and np.isfinite(probe).all()
assert (probe[:, DIM:] >= 0).all(), "Standard deviations must be non-negative"
print("Embedding shape per clip:", probe.shape, "(layers incl. CNN output, [mean | std])")

Embedding shape per clip: (25, 2048) (layers incl. CNN output, [mean | std])


In [6]:
from tqdm.auto import tqdm

embeddings, errors = [], []
for row in tqdm(manifest.itertuples(index=False), total=len(manifest), desc="WavLM"):
    try:
        embeddings.append(embed_audio(row.path))
    except Exception as e:
        errors.append((row.split, row.file_id, f"{type(e).__name__}: {e}"))
        embeddings.append(None)

if errors:
    display(pd.DataFrame(errors, columns=["split", "file_id", "error"]))
assert not errors, f"{len(errors)} files failed; see the table above"

WavLM:   0%|          | 0/985 [00:00<?, ?it/s]

## 4. Save and verify

In [7]:
emb = np.stack(embeddings).astype(np.float16)  # (files, layers + 1, 2 * dim); float16 halves the file size
assert emb.shape[0] == len(manifest) and np.isfinite(emb).all()

np.savez(OUT_PATH,
         split=manifest["split"].to_numpy(dtype=str),
         file_id=manifest["file_id"].to_numpy(dtype=str),
         emb=emb)

# Round-trip check: reload and compare.
check = np.load(OUT_PATH)
assert check["emb"].shape == emb.shape and np.array_equal(check["emb"], emb)
assert check["file_id"].tolist() == manifest["file_id"].tolist()
assert check["split"].tolist() == manifest["split"].tolist()

# check: every clip should have a different embedding.
flat = emb[:, -1, :].astype(np.float32)
n_dupes = len(flat) - len(np.unique(flat.round(3), axis=0))
print(f"Saved {OUT_PATH.name}: {emb.shape} ({OUT_PATH.stat().st_size / 1e6:.1f} MB), duplicate embeddings: {n_dupes}")

Saved audio_embeddings_wavlm_large.npz: (985, 25, 2048) (100.9 MB), duplicate embeddings: 0
